In [1]:
import sys
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.getcwd())

from src.models import CNN
from src.train import train
from src.dataset import HAM10000Dataset, get_train_transforms, get_eval_transforms, compute_class_weights, IMAGENET_MEAN, IMAGENET_STD, DX_TO_IDX, IDX_TO_DX
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader

Read in training and validation data, initialize HAM10000Dataset objects for each

In [2]:
train_df = pd.read_csv('data/splits/train.csv')
val_df = pd.read_csv('data/splits/val.csv')

image_dir = os.path.join(os.getcwd(), 'data', 'HAM10000_images')

train_dataset = HAM10000Dataset(train_df, image_dir, get_train_transforms())
val_dataset = HAM10000Dataset(val_df, image_dir, get_eval_transforms())

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=2)

Create CNN

In [3]:
cnn = CNN(
   in_channels=3,
   num_classes=7,
   n_blocks=4,
   n_layers_per_block=6,
   channels_l0=16,
   stem_kernel_size=11,
   stem_stride=2,
   block_stride=2,
   channel_growth_factor=2,
   dropout_p=0.3 
)

Construct loss function

In [4]:
class_weights = compute_class_weights(train_df)

loss_func = torch.nn.CrossEntropyLoss(weight=class_weights)

Construct Optimizer

In [5]:
lr = 1e-3
momentum = 0.9
adaptive_learning_rate = 0.999
betas = (momentum, adaptive_learning_rate)
optimizer = torch.optim.AdamW(cnn.parameters(), lr=lr, betas=betas)

Train CNN

In [6]:
cnn = train(
    model=cnn,
    model_name="baseline_cnn",
    train_loader=train_loader,
    val_loader=val_loader,
    loss_func=loss_func,
    optimizer=optimizer,
    num_epoch=50
)

Using CUDA
Epoch  1 / 50: train_acc=0.2985 val_acc=0.4940 train_f1=0.1605 val_f1=0.1459 
Epoch 10 / 50: train_acc=0.4562 val_acc=0.5387 train_f1=0.2758 val_f1=0.3108 
Epoch 20 / 50: train_acc=0.5277 val_acc=0.5387 train_f1=0.3447 val_f1=0.3624 
Epoch 30 / 50: train_acc=0.5758 val_acc=0.6306 train_f1=0.3992 val_f1=0.4224 
Epoch 40 / 50: train_acc=0.6022 val_acc=0.6382 train_f1=0.4404 val_f1=0.4871 
Epoch 50 / 50: train_acc=0.6137 val_acc=0.5972 train_f1=0.4641 val_f1=0.4763 
Model saved to logs\baseline_cnn_0708_144358\baseline_cnn_0708_144358.th
